# 2장 · 아껴 쓰는 비트 — 압축의 원리 📦

웹 챕터 **"아껴 쓰는 비트"** 와 짝을 이루는 Python 실습입니다.
웹의 코드 공방에서 잡은 감(자주 나오면 짧게)을, 여기서는 **허프만 코드를 직접 만들어**
실제 문장을 압축하며 굳힙니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch2')

## 1. 고정 길이 vs 가변 길이: 직접 세어 보기

A, B, C, D 가 확률 1/2, 1/4, 1/8, 1/8 로 나오는 메시지 1000글자를 만들어,
두 방식의 총 비트 수를 비교합니다.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)  # 결과 재현용 시드

letters = ['A', 'B', 'C', 'D']
probs = [1/2, 1/4, 1/8, 1/8]

message = ''.join(rng.choice(letters, size=1000, p=probs))
print('메시지 앞 60글자:', message[:60])

fixed_code = {'A': '00', 'B': '01', 'C': '10', 'D': '11'}          # 고정 2비트
var_code = {'A': '0', 'B': '10', 'C': '110', 'D': '111'}          # 자주 나오면 짧게

fixed_bits = sum(len(fixed_code[ch]) for ch in message)
var_bits = sum(len(var_code[ch]) for ch in message)

print(f'\n고정 길이: {fixed_bits}비트')
print(f'가변 길이: {var_bits}비트  (글자당 평균 {var_bits/1000:.3f}비트)')
print(f'절약: {fixed_bits - var_bits}비트 ({(fixed_bits - var_bits)/fixed_bits*100:.1f}%)')

글자당 평균이 이론값 **1.75비트**(= ½×1 + ¼×2 + ⅛×3 + ⅛×3) 근처로 나오죠.

**해독도 되는지** 확인해 봅시다. 접두어 코드라 구분자 없이 왼쪽부터 읽으면 됩니다.

In [ ]:
def decode(bits, code):
    """접두어 코드 해독 — 왼쪽부터 읽다가 아는 코드가 완성되는 순간 끊는다."""
    reverse = {v: k for k, v in code.items()}
    out, buf = [], ''
    for b in bits:
        buf += b
        if buf in reverse:
            out.append(reverse[buf])
            buf = ''
    return ''.join(out)

encoded = ''.join(var_code[ch] for ch in message)
decoded = decode(encoded, var_code)

print('비트열 앞 40비트:', encoded[:40])
print('해독 결과가 원본과 일치?', decoded == message)

## 2. 허프만 코드: 12줄로 만드는 최적 코드

규칙은 하나뿐입니다: **가장 드문 두 묶음을 합치면서, 합쳐지는 글자들의 코드 앞에 0/1 을 붙인다.**
이걸 묶음이 하나 남을 때까지 반복하면 끝. (웹 코드 공방이 실시간으로 하던 그 일)

In [ ]:
def huffman(freqs):
    """{글자: 빈도} → {글자: 코드}. 가장 드문 두 묶음을 반복해서 합친다."""
    codes = {ch: '' for ch in freqs}
    nodes = [(w, [ch]) for ch, w in freqs.items()]   # (묶음의 빈도 합, 소속 글자들)
    while len(nodes) > 1:
        nodes.sort(key=lambda n: n[0])
        (w1, group1), (w2, group2) = nodes.pop(0), nodes.pop(0)
        for ch in group1:
            codes[ch] = '0' + codes[ch]
        for ch in group2:
            codes[ch] = '1' + codes[ch]
        nodes.append((w1 + w2, group1 + group2))
    return codes

freqs = {'A': 8, 'B': 4, 'C': 2, 'D': 2}   # 1/2, 1/4, 1/8, 1/8 비율
codes = huffman(freqs)

total = sum(freqs.values())
print('글자 | 확률  | 허프만 코드 | 길이 | 놀람 log2(1/p)')
print('-' * 52)
for ch in 'ABCD':
    p = freqs[ch] / total
    print(f'  {ch}  | {p:.3f} | {codes[ch]:>10s} | {len(codes[ch])}비트 | {np.log2(1/p):.2f}비트')

**코드 길이와 놀람이 정확히 일치**합니다. 확률이 2의 거듭제곱의 역수(1/2, 1/4, 1/8)라서
허프만이 이론적 바닥에 딱 붙은 경우예요. 이상적 코드 길이 = 그 글자의 놀람!

## 3. 압축의 한계: 평균 놀람이라는 바닥

분포를 이것저것 바꿔 가며 **허프만 평균 길이**와 **평균 놀람** $\sum p \cdot \log_2(1/p)$ 을
나란히 놓아 봅니다. 허프만이 아무리 잘해도 바닥 아래로는 절대 못 내려가는 걸 확인하세요.

In [ ]:
def avg_code_length(freqs):
    codes = huffman(freqs)
    total = sum(freqs.values())
    return sum(w / total * len(codes[ch]) for ch, w in freqs.items())

def avg_surprise(freqs):
    total = sum(freqs.values())
    return sum(w / total * np.log2(total / w) for w in freqs.values())

distributions = {
    '균등 (25%씩)      ': {'A': 8, 'B': 8, 'C': 8, 'D': 8},
    '살짝 치우침       ': {'A': 16, 'B': 8, 'C': 4, 'D': 4},
    '극단 (A 천하)     ': {'A': 29, 'B': 1, 'C': 1, 'D': 1},
    '어중간 (딱 안 맞음)': {'A': 10, 'B': 7, 'C': 5, 'D': 3},
}

print('분포                | 허프만 평균 | 평균 놀람(바닥) | 바닥에 붙었나')
print('-' * 66)
for name, f in distributions.items():
    avg, floor = avg_code_length(f), avg_surprise(f)
    stuck = '딱 붙음 ✨' if abs(avg - floor) < 1e-9 else f'+{avg - floor:.3f}비트 손해'
    print(f'{name} | {avg:9.3f}비트 | {floor:11.3f}비트 | {stuck}')
    assert avg >= floor - 1e-9, '허프만이 바닥을 뚫었다?! (그럴 수 없음)'

print('\n어떤 분포에서도 허프만 평균 ≥ 평균 놀람. 이 바닥의 이름이 3장의 주인공, 엔트로피입니다.')

## 4. 실전: 진짜 영어 문장을 압축해 보기

이번엔 장난감 알파벳이 아니라 **실제 문장**입니다. 글자 빈도를 문장에서 직접 세서
허프만 코드를 만들고, 컴퓨터의 기본 방식(글자당 8비트 ASCII)과 비교합니다.

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# 한글 폰트 설정 (안 하면 한글 제목/라벨이 네모로 깨짐)
# Windows: 'Malgun Gothic' / macOS: 'AppleGothic' / Linux: 'NanumGothic' 로 바꾸세요
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False  # 음수 부호(-) 깨짐 방지

text = ('information theory studies the quantification storage and communication '
        'of information the field was established by claude shannon in the forties')

freqs = dict(Counter(text))
codes = huffman(freqs)

ascii_bits = len(text) * 8
huff_bits = sum(len(codes[ch]) for ch in text)
print(f'글자 수      : {len(text)}')
print(f'ASCII (8비트) : {ascii_bits}비트')
print(f'허프만        : {huff_bits}비트  (글자당 평균 {huff_bits/len(text):.2f}비트)')
print(f'절약          : {(ascii_bits - huff_bits)/ascii_bits*100:.0f}%')

# 가장 흔한 글자 8개: 빈도와 코드 길이가 반대로 가는 걸 눈으로
common = sorted(freqs, key=freqs.get, reverse=True)[:8]
labels = [repr(ch) for ch in common]
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.bar(labels, [freqs[ch] for ch in common], color='#6366f1', alpha=0.85, label='빈도 (회)')
ax1.set_ylabel('빈도 (회)', color='#6366f1')
ax2 = ax1.twinx()
ax2.plot(labels, [len(codes[ch]) for ch in common], 's-', color='#f59e0b', lw=2, ms=8, label='코드 길이 (비트)')
ax2.set_ylabel('허프만 코드 길이 (비트)', color='#f59e0b')
ax1.set_title('자주 나오면 짧게: 빈도(막대)가 클수록 코드 길이(선)가 짧다')
plt.show()

이게 zip 파일의 뼈대이고, GPT 의 **BPE 토큰화**도 같은 원리입니다.
자주 나오는 글자 뭉치("the", "ing")에 토큰 한 칸을 주니, 흔한 단어일수록 적은 토큰으로 끝나죠.

## 5. 직접 풀어보기 ✏️

확률이 A: 1/2, B: 1/4, C: 1/8, D: 1/8 이고 코드 길이가 1, 2, 3, 3 비트일 때,
**평균 코드 길이**는 몇 비트일까요? (웹 챕터의 어려움 문제와 같은 계산)

In [ ]:
# TODO: 아래 answer 를 여러분의 계산으로 바꾸고 실행하세요.
# 힌트: 평균 = Σ (확률 × 코드 길이)

answer = 1/2 * 1 + 1/4 * 2 + 1/8 * 3 + 1/8 * 3
print('내 답:', answer)
assert abs(answer - 1.75) < 1e-9, '다시 계산해 보세요 — 0.5 + 0.5 + 0.375 + 0.375'
print('✅ 잘했어요! 이 1.75비트는 이 분포의 평균 놀람과 같아서, 더는 줄일 수 없는 바닥입니다.')

---
**오늘의 한 줄**: 자주 나오면 짧게. 이상적 코드 길이는 그 글자의 놀람 log₂(1/p),
평균 놀람 아래로는 어떤 압축도 못 내려간다.

다음 챕터에서는 그 "평균 놀람"에 이름을 붙입니다. 이 과정의 핵심인 **엔트로피 H**. 🙂